# 03 Gold Layer — Aggregate & Business Metrics

**役割：** Silver層のクレンジング済みデータからビジネス指標を計算し、Streamlitが直接消費するテーブルを作る。  
**設計原則：** このレイヤーにビジネスロジックを集約する。Silver層には手を入れない。

---

### このノートブックの処理フロー
```
Delta Lake (Silver)
      │ Unity Catalog Table
      │ workspace.portfolio.silver_stock_returns
      │
      │ is_valid=True のみ使用
      ├─── 1. gold_stats          銘柄別：年率リターン・ボラティリティ・シャープレシオ
      ├─── 2. gold_corr_matrix    銘柄間相関行列
      └─── 3. gold_portfolio      加重後ポートフォリオ指標
                    │
                    ▼
             Delta Lake (Gold) ─── Unity Catalog Table
                    │ workspace.portfolio.gold_stats
                    │ workspace.portfolio.gold_corr_matrix
                    │ workspace.portfolio.gold_portfolio_metrics
                    │
                    ▼
             異常値分布の確認（±20%閾値の妥当性検証）
```

### 設計判断メモ
- **is_valid=True フィルタの位置**：Silver層ではなくGold層で行う。Silver層は判断しない設計のため
- **年率換算の係数**：営業日ベースで252日を使用。既存Streamlitアプリと統一
- **シャープレシオのリスクフリーレート**：設定値として外出し管理（現在2%想定）
- **相関行列の計算**：pandasのcorr()をSparkで再現するためにpivotを使う

## 0. ライブラリのインポート

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from datetime import datetime, timezone
import json

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

## 1. 設定値（Config）

In [0]:
# ── パス設定 ───────────────────────────────────────────────────
SILVER_TABLE         = "workspace.portfolio.silver_stock_returns"
GOLD_STATS_TABLE     = "workspace.portfolio.gold_stats"
GOLD_CORR_TABLE      = "workspace.portfolio.gold_corr_matrix"
GOLD_PORTFOLIO_TABLE = "workspace.portfolio.gold_portfolio_metrics"

# ── ビジネスパラメータ ─────────────────────────────────────────
TRADING_DAYS    = 252          # 年間営業日数
RISK_FREE_RATE  = 0.02         # リスクフリーレート（年率）：米10年債利回り想定

# ── ポートフォリオ設定（既存Streamlitアプリと同一） ────────────
TICKERS = ["SPY", "QQQ", "VT", "1540.T"]
WEIGHTS = {"SPY": 0.35, "QQQ": 0.40, "VT": 0.20, "1540.T": 0.05}

CALC_DATE    = datetime.now(timezone.utc).date()
PROCESSED_AT = datetime.now(timezone.utc)

print(f"リスクフリーレート : {RISK_FREE_RATE * 100}%")
print(f"ポートフォリオ比率 : {WEIGHTS}")
print(f"計算基準日         : {CALC_DATE}")

## 2. Silver層からデータ読み込み（is_valid=True のみ）

ここで初めて異常値フラグを使って除外する。Silver層の設計判断通り、判断はGold層が行う。

In [0]:
sdf_silver = (
    spark.read.table(SILVER_TABLE)
    .filter(F.col("is_valid") == True)
    .filter(F.col("daily_return").isNotNull())
)
display(sdf_silver.groupBy("ticker").count().orderBy("ticker"))

## 3. 異常値分布の確認（±20%閾値の妥当性検証）

Silver層で付与した`is_valid`フラグの分布を確認する。  
異常値として除外されたレコードが何件あるか、またその日次リターンの実際の値を確認する。

In [0]:
sdf_all = spark.read.table(SILVER_TABLE)

display(
    sdf_all
    .groupBy("ticker", "is_valid")
    .count()
    .orderBy("ticker", "is_valid")
)

display(
    sdf_all
    .filter(F.col("is_valid") == False)
    .select("ticker", "date", "adj_close", "daily_return")
    .orderBy(F.abs(F.col("daily_return")).desc())
    .limit(20)
)
# 確認ポイント：
# - 異常値が0件 → 過去10年のデータに±20%超の変動はなかった（正常）
# - 異常値が数件 → 日付・銘柄・値を確認してデータ品質問題か実際の極端相場か判断する

## 4. gold_stats — 銘柄別統計

既存Streamlitアプリの以下のロジックをSpark SQLに移管する：
```python
# 既存アプリ（pandas）
mu  = log_returns.mean() * 252          # 年率期待リターン
cov = log_returns.cov()  * 252          # 年率共分散行列
```

**pandas との対応：**
```python
# pandas
df.groupby('ticker')['daily_return'].mean() * 252

# Spark
sdf.groupBy('ticker').agg(F.mean('daily_return') * 252)
```

In [0]:
sdf_stats = (
    sdf_silver
    .groupBy("ticker")
    .agg(
        # 年率期待リターン：日次平均 × 252
        (F.mean("daily_return") * TRADING_DAYS).alias("annual_return"),

        # 年率ボラティリティ：日次標準偏差 × √252
        (F.stddev("daily_return") * F.sqrt(F.lit(TRADING_DAYS))).alias("annual_volatility"),

        F.min("date").alias("start_date"),
        F.max("date").alias("end_date"),
        F.count("*").alias("sample_days"),
    )
    # シャープレシオ：(年率リターン - リスクフリーレート) / 年率ボラティリティ
    .withColumn(
        "sharpe_ratio",
        (F.col("annual_return") - F.lit(RISK_FREE_RATE)) / F.col("annual_volatility")
    )
    .withColumn("calc_date",    F.lit(str(CALC_DATE)))
    .withColumn("processed_at", F.lit(PROCESSED_AT).cast(TimestampType()))
)

print("=== gold_stats（銘柄別統計） ===")
display(
    sdf_stats
    .select("ticker", "annual_return", "annual_volatility", "sharpe_ratio", "sample_days")
    .orderBy("ticker")
)
# 確認ポイント：
# - SPYの annual_return が概ね 0.10〜0.15（10〜15%）程度
# - sharpe_ratio が概ね 0.5〜1.0 程度（既存Streamlitアプリの値と近いか確認）

## 5. gold_corr_matrix — 銘柄間相関行列

Sparkには`pandas.corr()`に相当する直接的なメソッドがない。  
pivot → 相関計算 → unpivot（melt）の手順で再現する。

**pandas との対応：**
```python
# pandas（既存アプリ）
log_returns.corr()

# Spark：wide形式に変換してから相関を計算し、long形式に戻す
```

In [0]:
# Spark の相関計算は2列間のペアごとに行う
# 全銘柄ペアの組み合わせを列挙して計算する

corr_records = []

# 銘柄ごとのリターン列を別々に取得してpandasで相関行列を計算
# （件数が小さいためpandasへの変換が現実的）
pdf_returns = (
    sdf_silver
    .select("ticker", "date", "daily_return")
    .toPandas()                                      # Spark → pandas に変換
    .pivot(index="date", columns="ticker", values="daily_return")
    .dropna()                                        # 全銘柄が揃っている日だけ使用
)

# 相関行列を計算
corr_matrix = pdf_returns.corr()

print("=== 相関行列（pandas）===")
print(corr_matrix.round(4))

# long形式に変換してSparkに戻す
corr_long = (
    corr_matrix
    .reset_index()
    .melt(id_vars="ticker", var_name="ticker_b", value_name="correlation")
    .rename(columns={"ticker": "ticker_a"})
)
corr_long["calc_date"] = str(CALC_DATE)

sdf_corr = spark.createDataFrame(corr_long)

print("\n=== gold_corr_matrix（long形式） ===")
display(sdf_corr.orderBy("ticker_a", "ticker_b"))
# 確認ポイント：
# - 対角成分（ticker_a = ticker_b）は 1.0
# - SPY-QQQ は高相関（0.85以上）が想定される
# - 1540.T（金）は他と低相関（分散効果）が想定される

## 6. gold_portfolio_metrics — ポートフォリオ加重後指標

銘柄別統計と相関行列を使って、ポートフォリオ全体の指標を計算する。

**計算式：**
```
ポートフォリオ期待リターン = Σ(w_i × μ_i)
ポートフォリオ分散        = Σ_i Σ_j (w_i × w_j × σ_i × σ_j × ρ_ij)
ポートフォリオリスク      = √(分散)
```
これは既存Streamlitアプリが行っていた計算と同一ロジック。

In [0]:
import numpy as np

# gold_stats から銘柄別の統計量を取得
stats_dict = {
    row["ticker"]: {
        "annual_return":     row["annual_return"],
        "annual_volatility": row["annual_volatility"],
    }
    for row in sdf_stats.collect()
}

# ウェイト・リターン・ボラティリティをWEIGHTS順に整列
tickers = TICKERS
weights = np.array([WEIGHTS[t] for t in tickers])
returns = np.array([stats_dict[t]["annual_return"]     for t in tickers])
vols    = np.array([stats_dict[t]["annual_volatility"] for t in tickers])

# ポートフォリオ期待リターン
portfolio_return = float(np.dot(weights, returns))

# 相関行列から共分散行列を構築
corr_pivot = corr_matrix.loc[tickers, tickers].values   # 銘柄順を統一
cov_matrix = np.outer(vols, vols) * corr_pivot          # σ_i × σ_j × ρ_ij

# ポートフォリオリスク（標準偏差）
portfolio_variance = float(weights @ cov_matrix @ weights)
portfolio_risk     = float(np.sqrt(portfolio_variance))

# シャープレシオ
portfolio_sharpe = (portfolio_return - RISK_FREE_RATE) / portfolio_risk

print("=== ポートフォリオ加重後指標 ===")
print(f"期待リターン（年率）: {portfolio_return * 100:.2f}%")
print(f"リスク（年率）      : {portfolio_risk * 100:.2f}%")
print(f"シャープレシオ      : {portfolio_sharpe:.3f}")
# 参考：既存アプリの表示値
# 期待リターン 15.59% / リスク 18.07% / シャープレシオ 0.86

## 7. Gold層への書き込み

In [0]:
sdf_stats.write.format("delta").mode("overwrite").saveAsTable(GOLD_STATS_TABLE)
print(f"✅ gold_stats 書き込み完了: {GOLD_STATS_TABLE}")

sdf_corr.write.format("delta").mode("overwrite").saveAsTable(GOLD_CORR_TABLE)
print(f"✅ gold_corr_matrix 書き込み完了: {GOLD_CORR_TABLE}")

import pandas as pd
pdf_portfolio = pd.DataFrame([{
    "expected_return" : portfolio_return,
    "portfolio_risk"  : portfolio_risk,
    "sharpe_ratio"    : portfolio_sharpe,
    "weights"         : json.dumps(WEIGHTS),
    "calc_date"       : str(CALC_DATE),
}])
sdf_portfolio = spark.createDataFrame(pdf_portfolio)
sdf_portfolio.write.format("delta").mode("overwrite").saveAsTable(GOLD_PORTFOLIO_TABLE)
print(f"✅ gold_portfolio_metrics 書き込み完了: {GOLD_PORTFOLIO_TABLE}")

## 8. 書き込み後の確認

In [0]:
print("=== gold_stats ===")
display(spark.read.table(GOLD_STATS_TABLE).orderBy("ticker"))
print("=== gold_corr_matrix ===")
display(spark.read.table(GOLD_CORR_TABLE).orderBy("ticker_a", "ticker_b"))
print("=== gold_portfolio_metrics ===")
display(spark.read.table(GOLD_PORTFOLIO_TABLE))

---

## ✅ Gold層 完了チェックリスト

- [ ] 異常値レコード（is_valid=False）の件数と内容を確認した
- [ ] SPYの annual_return が概ね 10〜15% の範囲に収まっている
- [ ] シャープレシオが概ね 0.5〜1.0 の範囲に収まっている
- [ ] 相関行列の対角成分がすべて 1.0 になっている
- [ ] SPY-QQQ の相関が 0.85 以上になっている
- [ ] ポートフォリオ指標が既存Streamlitアプリの値と近似している（Close→adj_close変更による微差は許容）
- [ ] 3テーブルすべてUnity Catalog TableとしてGold層への書き込みが完了した
      （workspace.portfolio.gold_stats / gold_corr_matrix / gold_portfolio_metrics）

---

**次のステップ：** `04_export_job.ipynb` — Gold層のデータをCSV/Parquetにエクスポートし、Streamlitが読み込める形に変換する